In [1]:
import pandas as pd
import numpy as np

# Daten herunterladen
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

# Spalten filtern
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols].copy()

# Question 1: Missing Values
print("--- Question 1 ---")
print(df.isnull().sum())

# Question 2: Median horsepower
print("\n--- Question 2 ---")
print("Median horsepower:", df['horsepower'].median())

--- Question 1 ---
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

--- Question 2 ---
Median horsepower: 254.0


In [3]:
def prepare_split(df_data, seed=42):
    n = len(df_data)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df_data.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df_data.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df_data.iloc[idx[n_train + n_val:]].reset_index(drop=True)

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values
    y_test = df_test['fuel_efficiency_mpg'].values

    del df_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']

    return df_train, df_val, df_test, y_train, y_val, y_test

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    idx = np.eye(XTX.shape[0])
    XTX = XTX + r * idx
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y_true, y_pred):
    error = y_true - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [5]:
df_train, df_val, df_test, y_train, y_val, y_test = prepare_split(df, seed=42)

# Option 1: Mit 0 auffüllen
X_train_0 = df_train.fillna(0).values
w0_0, w_0 = train_linear_regression(X_train_0, y_train)

X_val_0 = df_val.fillna(0).values
y_pred_0 = w0_0 + X_val_0.dot(w_0)
score_0 = round(rmse(y_val, y_pred_0), 3)

# Option 2: Mit Mean des TRAIN-DATENSATZES auffüllen
mean_hp = df_train['horsepower'].mean()

X_train_mean = df_train.fillna(mean_hp).values
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

X_val_mean = df_val.fillna(mean_hp).values
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
score_mean = round(rmse(y_val, y_pred_mean), 3)

print("--- Question 3 ---")
print("RMSE (fillna 0):   ", score_0)
print("RMSE (fillna mean):", score_mean)

--- Question 3 ---
RMSE (fillna 0):    2.205
RMSE (fillna mean): 2.202


In [6]:
df_train, df_val, df_test, y_train, y_val, y_test = prepare_split(df, seed=42)

X_train = df_train.fillna(0).values
X_val = df_val.fillna(0).values

print("--- Question 4 ---")
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r={r:<5} | RMSE={score}")

--- Question 4 ---
r=0     | RMSE=2.2053
r=0.01  | RMSE=2.2058
r=0.1   | RMSE=2.2241
r=1     | RMSE=2.3492
r=5     | RMSE=2.4094
r=10    | RMSE=2.4195
r=100   | RMSE=2.4292


In [7]:
scores = []

for s in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    df_train, df_val, df_test, y_train, y_val, y_test = prepare_split(df, seed=s)
    
    X_train = df_train.fillna(0).values
    X_val = df_val.fillna(0).values
    
    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    
    score = rmse(y_val, y_pred)
    scores.append(score)

std_val = round(np.std(scores), 3)

print("--- Question 5 ---")
print("Scores:", [round(s, 3) for s in scores])
print("Standardabweichung:", std_val)

--- Question 5 ---
Scores: [np.float64(2.239), np.float64(2.202), np.float64(2.163), np.float64(2.204), np.float64(2.202), np.float64(2.246), np.float64(2.27), np.float64(2.191), np.float64(2.217), np.float64(2.207)]
Standardabweichung: 0.029


In [8]:
# 1. Split mit Seed 9
df_train, df_val, df_test, y_train, y_val, y_test = prepare_split(df, seed=9)

# 2. Train und Validation kombinieren
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)
y_full_train = np.concatenate([y_train, y_val])

# 3. Fehlwerte mit 0 auffüllen
X_full_train = df_full_train.fillna(0).values
X_test = df_test.fillna(0).values

# 4. Modifizierte Ridge Regression mit r=0.001 trainieren
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# 5. Auf Testdaten evaluieren
y_pred_test = w0 + X_test.dot(w)
test_score = round(rmse(y_test, y_pred_test), 3)

print("--- Question 6 ---")
print("Test RMSE:", test_score)

--- Question 6 ---
Test RMSE: 2.236
